# M and A, workshop 1 — Three files, three dialects

**The situation.** Your company just bought a competitor. Their engineer left, and his parting
words were: *"at least we had stored all the information"*. You open the export and find three
customer files that disagree about almost everything — separators, column order, capitalisation,
even whether there is a header.

Your job is to make them one table. Today is step one, and it is the step everyone skips:

1. **Look at the bytes before you parse.**
2. **Write the target schema down first.**
3. **Load each file into a DataFrame that is actually correct.**

We work on six-row miniatures of the real files — same diseases, small enough to read with your
eyes. Everything you build here applies unchanged to the full exports.

In [ ]:
# ABOUTME: M and A workshop 1 - reading three incompatible customer exports.
# ABOUTME: File dialects, pd.read_csv arguments, and mapping source columns onto a target schema.
import csv
import pandas as pd

pd.set_option("display.width", 150)
pd.set_option("display.max_columns", 20)

In [ ]:
from pathlib import Path

WORK = Path("sample_files")      # three little files, written next to this notebook
WORK.mkdir(exist_ok=True)

# --- Shop A: a normal CSV export. Quoted names, a placeholder email, a country typed as a number.
(WORK / "shop_a.csv").write_text(
    "Gender,FirstName,LastName,UserName,Email,Age,City,Country\n"
    "Female,latesha,Barton,latesha,forgottoask@woodinc,85,INDIANAPOLIS,United State Of America\n"
    'Female,Yolando,"Jast",yolando,yolando_jast@yahoo.com,53,Columbus,12\n'
    "Male,BRANDON,Hyatt,brandon,brandon.hyatt@gmail.com,41,new_york,U.S.A.\n"
    "M,chase,OBERBRUNNER,chase,CHASE.OBERBRUNNER@HOTMAIL.COM,29,Austin,US\n"
    "0,Erwin,Kuhic,erwin,forgottoask@woodinc,66,Phoenix,USA\n"
    "1,Tawanda,Lind,tawanda,tawanda.lind@yahoo.com,37,Seattle,U.S.\n"
)

# --- Shop B: no header, ';' separated, columns in another order, and no country column at all.
(WORK / "shop_b.csv").write_text(
    '"51";austin;1;"lorette" kutch;Kutch.lorette@gmail.com\n'
    '"84";san diego;Female;CLORINDA WIZA;clorinda.wiza@gmail.com\n'
    '"97";Columbus;Male;"\\"Joesph\\"" "littel";LITTEL.JOESPH@GMAIL.COM\n'
    '"23";New_York;0;carlo Stokes;carlo.stokes@hotmail.com\n'
    '"45";Chicago;M;KASEY "prosacco";kasey_prosacco@gmail.com\n'
    '"61";Dallas;F;brandon hyatt;brandon.hyatt@gmail.com\n'
)

# --- Shop C: header names comma separated, rows TAB separated, every value carrying its type.
(WORK / "shop_c.csv").write_text(
    "Gender,Name,Email,Age,City,Country\n"
    'string_Male\tstring_kendall DACH\tstring_DACH.KENDALL@HOTMAIL.COM\t"integer_26"\tstring_Dallas\tstring_United_State_Of_America\n'
    'string_Male\tstring_jonathon "schneider"\tstring_Jonathon.schneider@gmail.com\t"integer_91"\tstring_INDIANAPOLIS\tstring_Us\n'
    'boolean_1\tstring_JAROD konopelski\tstring_KONOPELSKI_JAROD@HOTMAIL.COM\t"integer_38"\tstring_Chicago\tstring_1\n'
    'character_M\tstring_marta Bergstrom\tstring_marta.bergstrom@yahoo.com\t"integer_52"\tstring_Phoenix\tstring_u.s.a.\n'
    'boolean_0\tstring_SHERYL hamill\tstring_sheryl.hamill@gmail.com\t"integer_19"\tstring_Austin\tstring_US\n'
    'string_Female\tstring_lorette kutch\tstring_Kutch.lorette@gmail.com\t"integer_51"\tstring_Austin\tstring_U.S.\n'
)

print("wrote", ", ".join(sorted(p.name for p in WORK.iterdir())))

wrote shop_a.csv, shop_b.csv, shop_c.csv


## 1. Look before you parse

`pd.read_csv` will happily give you a DataFrame for any of these. A wrong DataFrame, usually,
and it will not warn you. So first: print the raw text.

**Rule of thumb:** never call `read_csv` on a file whose first three lines you have not read.

In [ ]:
for name in ["shop_a.csv", "shop_b.csv", "shop_c.csv"]:
    print(f"----- {name}")
    print((WORK / name).read_text().split("\n")[0][:110])
    print((WORK / name).read_text().split("\n")[1][:110])
    print()

----- shop_a.csv
Gender,FirstName,LastName,UserName,Email,Age,City,Country
Female,latesha,Barton,latesha,forgottoask@woodinc,85,INDIANAPOLIS,United State Of America

----- shop_b.csv
"51";austin;1;"lorette" kutch;Kutch.lorette@gmail.com
"84";san diego;Female;CLORINDA WIZA;clorinda.wiza@gmail.com

----- shop_c.csv
Gender,Name,Email,Age,City,Country
string_Male	string_kendall DACH	string_DACH.KENDALL@HOTMAIL.COM	"integer_26"	string_Dallas	string_United_State



Three different worlds:

| file | separator | header | notable |
|---|---|---|---|
| **shop_a** | `,` | yes | looks normal — it is the trap |
| **shop_b** | `;` | **none** | columns in a different order, values quoted at random |
| **shop_c** | **tab** in the rows, `,` in the header line | sort of | every value prefixed with its type |

Watch what the naive call does to the last two.

In [ ]:
naive_b = pd.read_csv(WORK / "shop_b.csv")
print("shop_b naive:", naive_b.shape, "<- one column, and the first customer became the header")
display(naive_b.head(2))

naive_c = pd.read_csv(WORK / "shop_c.csv")
print("shop_c naive:", naive_c.shape, "<- six names in the header, one column of tab-joined values")
display(naive_c.head(2))

shop_b naive: (5, 1) <- one column, and the first customer became the header


,"51;austin;1;""lorette"" kutch;Kutch.lorette@gmail.com"
0,84;san diego;Female;CLORINDA WIZA;clorinda.wiz...
1,"97;Columbus;Male;""\""Joesph\"""" ""littel"";LITTEL...."


shop_c naive: (6, 6) <- six names in the header, one column of tab-joined values


,Gender,Name,Email,Age,City,Country
0,string_Male\tstring_kendall DACH\tstring_DACH....,NaN,NaN,NaN,NaN,NaN
1,"string_Male\tstring_jonathon ""schneider""\tstri...",NaN,NaN,NaN,NaN,NaN


Neither raised an error. Both are wrong. `shop_b` lost a customer into the header row;
`shop_c` reports six columns that contain nothing.

**The punchline:** silence is not success. After every load, check `.shape`, `.columns` and
`.head()` against what you saw in the raw text.

## 2. The arguments that fix dialects

`pd.read_csv` has one argument for each thing that can differ:

| problem | argument |
|---|---|
| separator is not a comma | `sep=";"` or `sep="\t"` |
| there is no header line | `header=None` plus `names=[...]` |
| there *is* a header you want to ignore | `skiprows=1` |
| quotes inside values confuse the parser | `quoting=csv.QUOTE_NONE` |
| blank-looking values are really missing | `na_values=[""]`, `keep_default_na=True` |

### Exercise 1 — load shop_b

No header, `;` separated. From the raw text, the column order is:
`age`, `city`, `gender`, `name`, `email`.

Load it into a DataFrame with those column names, 6 rows and 5 columns.

In [ ]:
def load_shop_b(path):
    """Return shop_b as a DataFrame with columns age, city, gender, name, email."""
    return pd.read_csv(path, sep=";", header=None,
                       names=["age", "city", "gender", "name", "email"])

shop_b = load_shop_b(WORK / "shop_b.csv")
shop_b

,age,city,gender,name,email
0,51,austin,1,lorette kutch,Kutch.lorette@gmail.com
1,84,san diego,Female,CLORINDA WIZA,clorinda.wiza@gmail.com
2,97,Columbus,Male,"\Joesph\"""" ""littel""",LITTEL.JOESPH@GMAIL.COM
3,23,New_York,0,carlo Stokes,carlo.stokes@hotmail.com
4,45,Chicago,M,"KASEY ""prosacco""",kasey_prosacco@gmail.com
5,61,Dallas,F,brandon hyatt,brandon.hyatt@gmail.com


In [ ]:
def check(label, *conditions):
    """Report an exercise self-check without raising: a red traceback is a bad first impression."""
    try:
        ok = all(bool(c()) for c in conditions)
    except Exception as exc:                      # a stub that still returns None lands here
        print(f"  {label}: not yet ({type(exc).__name__}: {exc})")
        return False
    print(f"  {label}: {'PASS' if ok else 'not yet'}")
    return ok

check("6 rows, 5 columns", lambda: shop_b.shape == (6, 5))
check("column names", lambda: list(shop_b.columns) == ["age", "city", "gender", "name", "email"])
check("no customer lost to the header", lambda: (shop_b.email == "Kutch.lorette@gmail.com").any())

  6 rows, 5 columns: PASS
  column names: PASS
  no customer lost to the header: PASS


True

### Exercise 2 — load shop_c

Harder. The first line names the columns, comma separated. Every line after it is **tab**
separated. And the values carry quotes that the parser will try to interpret.

Two steps: take the column names from line 1, then parse the rest as a tab-separated file.
Hint: `path.read_text().split("\n", 1)` splits a text into its first line and the rest, and
`io.StringIO(text)` turns a string into something `read_csv` can read.

In [ ]:
import io

def load_shop_c(path):
    """Return shop_c as a DataFrame with the 6 columns named in its first line."""
    raw = path.read_text()
    header, body = raw.split("\n", 1)
    names = header.strip().split(",")
    return pd.read_csv(io.StringIO(body), sep="\t", header=None, names=names,
                       quoting=csv.QUOTE_NONE)

shop_c = load_shop_c(WORK / "shop_c.csv")
shop_c

,Gender,Name,Email,Age,City,Country
0,string_Male,string_kendall DACH,string_DACH.KENDALL@HOTMAIL.COM,"""integer_26""",string_Dallas,string_United_State_Of_America
1,string_Male,"string_jonathon ""schneider""",string_Jonathon.schneider@gmail.com,"""integer_91""",string_INDIANAPOLIS,string_Us
2,boolean_1,string_JAROD konopelski,string_KONOPELSKI_JAROD@HOTMAIL.COM,"""integer_38""",string_Chicago,string_1
3,character_M,string_marta Bergstrom,string_marta.bergstrom@yahoo.com,"""integer_52""",string_Phoenix,string_u.s.a.
4,boolean_0,string_SHERYL hamill,string_sheryl.hamill@gmail.com,"""integer_19""",string_Austin,string_US
5,string_Female,string_lorette kutch,string_Kutch.lorette@gmail.com,"""integer_51""",string_Austin,string_U.S.


In [ ]:
def check(label, *conditions):
    """Report an exercise self-check without raising: a red traceback is a bad first impression."""
    try:
        ok = all(bool(c()) for c in conditions)
    except Exception as exc:                      # a stub that still returns None lands here
        print(f"  {label}: not yet ({type(exc).__name__}: {exc})")
        return False
    print(f"  {label}: {'PASS' if ok else 'not yet'}")
    return ok

check("6 rows, 6 columns", lambda: shop_c.shape == (6, 6))
check("column names from line 1", lambda: list(shop_c.columns) == ["Gender", "Name", "Email", "Age", "City", "Country"])
check("values still carry their type prefix", lambda: str(shop_c.Gender.iloc[0]).startswith("string_"))

  6 rows, 6 columns: PASS
  column names from line 1: PASS
  values still carry their type prefix: PASS


True

Note the last check: `shop_c` is now *correctly parsed* and still *completely unusable* —
`string_Male`, `"integer_26"`. Parsing and cleaning are two different jobs, and mixing them is
how loaders become unreadable. Cleaning is next workshop.

## 3. The target schema comes first

You cannot merge three tables into "whatever comes out". You decide the destination, then make
each source arrive there. Our warehouse already has a `customers` table:

```
gender, firstname, lastname, email, age, city, country, created_at, referral
```

Nine columns. Now map each source onto it.

### Exercise 3 — the mapping table

Fill the dictionary: for each target column, which source column provides it, per file.
Use `None` where a file simply does not have it.

In [ ]:
TARGET_SCHEMA = ["gender", "firstname", "lastname", "email", "age", "city", "country",
                 "created_at", "referral"]

mapping = {
    "shop_a": {"gender": "Gender", "firstname": "FirstName", "lastname": "LastName",
               "email": "Email", "age": "Age", "city": "City", "country": "Country",
               "created_at": None, "referral": None},
    # shop_b: one "name" column feeds two target columns, and there is no country at all.
    "shop_b": {"gender": "gender", "firstname": "name", "lastname": "name",
               "email": "email", "age": "age", "city": "city", "country": None,
               "created_at": None, "referral": None},
    "shop_c": {"gender": "Gender", "firstname": "Name", "lastname": "Name",
               "email": "Email", "age": "Age", "city": "City", "country": "Country",
               "created_at": None, "referral": None},
}
mapping

{'shop_a': {'gender': 'Gender',
  'firstname': 'FirstName',
  'lastname': 'LastName',
  'email': 'Email',
  'age': 'Age',
  'city': 'City',
  'country': 'Country',
  'created_at': None,
  'referral': None},
 'shop_b': {'gender': 'gender',
  'firstname': 'name',
  'lastname': 'name',
  'email': 'email',
  'age': 'age',
  'city': 'city',
  'country': None,
  'created_at': None,
  'referral': None},
 'shop_c': {'gender': 'Gender',
  'firstname': 'Name',
  'lastname': 'Name',
  'email': 'Email',
  'age': 'Age',
  'city': 'City',
  'country': 'Country',
  'created_at': None,
  'referral': None}}

In [ ]:
def check(label, *conditions):
    """Report an exercise self-check without raising: a red traceback is a bad first impression."""
    try:
        ok = all(bool(c()) for c in conditions)
    except Exception as exc:                      # a stub that still returns None lands here
        print(f"  {label}: not yet ({type(exc).__name__}: {exc})")
        return False
    print(f"  {label}: {'PASS' if ok else 'not yet'}")
    return ok

check("every file maps every target column",
      lambda: all(set(m) == set(TARGET_SCHEMA) for m in mapping.values()))
check("shop_b has no country", lambda: mapping["shop_b"]["country"] is None)
check("one name column feeds two targets",
      lambda: mapping["shop_b"]["firstname"] == mapping["shop_b"]["lastname"] != None)
check("nobody has created_at or referral",
      lambda: all(m["created_at"] is None and m["referral"] is None for m in mapping.values()))

  every file maps every target column: PASS
  shop_b has no country: PASS
  one name column feeds two targets: PASS
  nobody has created_at or referral: PASS


True

### Exercise 4 — what the mapping just told you

Three facts fall out of that table, and each is a decision you must make *before* writing merge
code. Answer in the cell below.

1. Which target columns does **no** source provide?
2. `shop_b` has no country. What will you put there?
3. `shop_a` has a `UserName` column that the target schema does not want. What happens to it?

In [ ]:
missing_everywhere = ["created_at", "referral"]
shop_b_country_plan = (
    "Leave it missing (NaN). Every customer in the file is American, so filling 'United States "
    "of America' is tempting - but that is an assumption about data we were never given, and a "
    "NaN we can explain beats a value we invented."
)
username_plan = (
    "Dropped. The target schema is the contract; a column it does not define has nowhere to go. "
    "Worth one line in the migration notes in case someone asks for it later."
)

print(missing_everywhere, "|", shop_b_country_plan, "|", username_plan)

['created_at', 'referral'] | Leave it missing (NaN). Every customer in the file is American, so filling 'United States of America' is tempting - but that is an assumption about data we were never given, and a NaN we can explain beats a value we invented. | Dropped. The target schema is the contract; a column it does not define has nowhere to go. Worth one line in the migration notes in case someone asks for it later.


In [ ]:
def check(label, *conditions):
    """Report an exercise self-check without raising: a red traceback is a bad first impression."""
    try:
        ok = all(bool(c()) for c in conditions)
    except Exception as exc:                      # a stub that still returns None lands here
        print(f"  {label}: not yet ({type(exc).__name__}: {exc})")
        return False
    print(f"  {label}: {'PASS' if ok else 'not yet'}")
    return ok

check("Exercise 4", lambda: sorted(missing_everywhere) == ["created_at", "referral"],
      lambda: len(shop_b_country_plan) > 20, lambda: len(username_plan) > 20)

  Exercise 4: PASS


True

There is no single right answer to 2 and 3 — there is a *defensible* one. In a migration, the
decisions you can explain six months later are worth more than the clever ones.

## 4. All three, side by side

In [ ]:
shop_a = pd.read_csv(WORK / "shop_a.csv")
for name, df in [("shop_a", shop_a), ("shop_b", shop_b), ("shop_c", shop_c)]:
    print(f"{name}: {df.shape[0]} rows x {df.shape[1]} cols | {', '.join(df.columns)}")

shop_a: 6 rows x 8 cols | Gender, FirstName, LastName, UserName, Email, Age, City, Country
shop_b: 6 rows x 5 cols | age, city, gender, name, email
shop_c: 6 rows x 6 cols | Gender, Name, Email, Age, City, Country


Three correct DataFrames. Not three *clean* ones — look at them and you will still find
`string_Male`, `forgottoask@woodinc`, a country called `12`, `INDIANAPOLIS` next to `new_york`,
and a gender column that uses six different encodings.

That is next week.

## What to take away

1. **Read the raw text first.** Two minutes of looking saves an afternoon of wrong numbers.
2. **A load that does not raise is not a load that worked.** Check shape, columns, and one row.
3. **`read_csv` has an argument for every dialect** — `sep`, `header`, `names`, `skiprows`,
   `quoting`. You are not supposed to memorise them, you are supposed to reach for them.
4. **Write the target schema before the merge code.** It turns a vague "combine these" into a
   checklist of per-column decisions.
5. **Parse first, clean second.** Separate functions, separate workshop.

### Take it home
Run this on the real exports: `uv run get_data.py` in the `workshops/` folder downloads the three
Only Wood files into `data/`. Load each one correctly and print its shape — the dialects are
exactly the ones you just handled, only with 10 000 rows each.